# SOC 시뮬레이션 — Volvo FH Electric, 부산 → 인천공항

## 이 노트북이 푸는 문제

배터리 100%에서 출발해 구간마다 SOC를 깎아가며, **어느 휴게소에서 충전해야 완주하는지** 계산합니다.

## 먼저 확인한 사실

`ev_energy_consumption.csv` 는 **이 트럭에 그대로 못 씁니다.**

| | 데이터셋 | FH Electric |
|---|---|---|
| 전비 | 11.6 ~ 35.0 (평균 24.1) kWh/100km | **97.9** kWh/100km |
| 적재량 | 0 ~ 500 kg | 최대 약 28,000 kg |

전비가 **4.1배** 차이납니다. 데이터셋 최대값(35)조차 트럭 공인치의 36%입니다.
적재량은 두 자릿수 배율로 벗어나고요. 외삽이 아니라 다른 차종입니다.

## 그래서 취한 방법 — 물리 모델 + ML 보정

```
전비_트럭 = 물리모델(질량, 속도, 구배)  ×  ML_보정계수
ML_보정계수 = ML예측(현재조건) / ML예측(기준조건)
```

- **물리 모델**이 절대 수준(질량·속도·구배)을 잡습니다. 종방향 동역학이라 트럭 질량에서도 성립합니다.
- **ML 모델**은 물리식이 잘 못 다루는 것들(기온, HVAC, 운전 스타일, 타이어압)의 **상대적 영향**만 씁니다.
  절대값이 아니라 비율로 쓰기 때문에 스케일 불일치를 우회합니다.

이 방식의 한계도 분명합니다: ML의 상대 민감도가 트럭에서도 같다고 **가정**하는 겁니다.
승용차와 트럭은 열관리 부하 비중이 달라 기온 민감도가 다를 수 있습니다.
실측 트럭 데이터가 생기면 물리 모델은 그대로 두고 보정계수만 재학습하세요.

## 0. 트럭 제원 및 상수

In [1]:
import numpy as np
import pandas as pd

# ── Volvo FH Electric 공개 제원 (volvotrucks.com) ──────────────────
BATT_TOTAL_KWH = 540.0     # 360~540 kWh, 4~6팩 중 최대 구성
BATT_USABLE_KWH = 460.0    # "Up to 460 kWh"
OEM_RANGE_KM = 470.0       # "Up to 470 km"
CHG_MAX_KW = 350.0         # CCS
CHG_20_80_MIN = 65.0

# ── 운행 조건 (여기를 바꿔가며 시나리오를 보세요) ──────────────────
GCW_KG = 40_000            # 총조합중량. 한국 도로법 상한 40t (EU 스펙은 65t)
SOC_START = 100.0          # 출발 SOC (%)
SOC_MIN = 20.0             # 이 밑으로 떨어지면 안 됨
SOC_CHARGE_TO = 80.0       # 급속충전 종료 SOC (80% 넘으면 충전속도 급락)
AMBIENT_C = 5.0            # 주행 시 외기온
DRIVING_STYLE = 0.35       # 0~1. 물류 정속주행이면 낮게

# ── 물리 상수 ──────────────────────────────────────────────────
G, RHO_AIR = 9.81, 1.2
CRR = 0.006                # 트럭 타이어 구름저항계수
CDA = 5.5                  # Cd×A (트랙터+트레일러, 약 5~6 m²)
ETA_DRIVE = 0.85           # 구동계 효율
ETA_REGEN = 0.60           # 회생제동 회수율
AUX_KW = 3.0               # 보조장치 기본 부하

oem_kwh100 = BATT_USABLE_KWH / OEM_RANGE_KM * 100
print(f"공인 전비 역산: {BATT_USABLE_KWH:.0f} kWh / {OEM_RANGE_KM:.0f} km = {oem_kwh100:.1f} kWh/100km")
print(f"20→80% = {BATT_USABLE_KWH * 0.6:.0f} kWh / {CHG_20_80_MIN:.0f}분 "
      f"= 평균 {BATT_USABLE_KWH * 0.6 / (CHG_20_80_MIN / 60):.0f} kW (피크의 "
      f"{BATT_USABLE_KWH * 0.6 / (CHG_20_80_MIN / 60) / CHG_MAX_KW * 100:.0f}%)")

공인 전비 역산: 460 kWh / 470 km = 97.9 kWh/100km
20→80% = 276 kWh / 65분 = 평균 255 kW (피크의 73%)


## 1. 물리 기반 전비 모델

$$P_{\text{wheel}} = \underbrace{C_{rr}\,m\,g\,v}_{\text{구름저항}} + \underbrace{\tfrac{1}{2}\rho\,C_dA\,v^3}_{\text{공기저항}} + \underbrace{m\,g\,\sin\theta\,v}_{\text{구배}}$$

내리막에서는 회생제동으로 일부를 회수합니다. 회수율이 100%가 아니라 **오르막·내리막이 비대칭**이라,
구간 평균 구배 하나로 뭉뚱그리면 안 됩니다. 상승분과 하강분을 따로 넣으세요.

In [2]:
def segment_energy_kwh(dist_km, v_kmh, ascent_m=0.0, descent_m=0.0,
                       mass_kg=GCW_KG, aux_kw=AUX_KW):
    """구간 총 소모 에너지(kWh).

    구배를 '구간 평균 %'로 넣으면 안 됩니다. 60 km 구간의 오르막과 내리막이
    평균에서 상쇄되어 사라지는데, 회생 회수율이 60%라 실제로는 상쇄되지 않습니다.
    그래서 상승 누적(ascent_m)과 하강 누적(descent_m)을 따로 받습니다.
    """
    v = v_kmh / 3.6
    d = dist_km * 1000

    f_roll = CRR * mass_kg * G                      # N
    f_aero = 0.5 * RHO_AIR * CDA * v ** 2           # N
    e_resist = (f_roll + f_aero) * d / ETA_DRIVE    # J

    e_climb = mass_kg * G * ascent_m / ETA_DRIVE    # 오르막: 손실 포함 전량 소모
    e_regen = mass_kg * G * descent_m * ETA_REGEN   # 내리막: 회수율만큼만 회수
    e_aux = aux_kw * 1000 * (dist_km / v_kmh * 3600)

    return max(e_resist + e_climb - e_regen + e_aux, 0.0) / 3.6e6


def kwh_per_100km(v_kmh, mass_kg=GCW_KG, **kw):
    """평지 정속 환산 전비 (비교·검증용)."""
    return segment_energy_kwh(100.0, v_kmh, mass_kg=mass_kg, **kw)


grid = pd.DataFrame(
    {v: [round(kwh_per_100km(v, m * 1000), 1) for m in [25, 30, 40, 65]] for v in [80, 90, 100]},
    index=[f"{m}t" for m in [25, 30, 40, 65]])
grid.columns.name, grid.index.name = "속도 km/h", "총중량"
print(f"평지 정속 전비 (kWh/100km)   ※ 공인치 {oem_kwh100:.0f}\n")
print(grid.to_string())
print(f"\n→ 공인 {oem_kwh100:.0f} kWh/100km 는 25~30t·80km/h 조건입니다.")
print(f"   40t 만재 90km/h 면 {kwh_per_100km(90, 40000):.0f} kWh/100km 로 봐야 합니다.")

# 고도 기복의 영향 — 순고도차 0이어도 전비는 오릅니다
flat = segment_energy_kwh(100, 90)
hilly = segment_energy_kwh(100, 90, ascent_m=500, descent_m=500)
print(f"\n100 km 구간, 순고도차 0:")
print(f"  완전 평지        {flat:.0f} kWh")
print(f"  ±500 m 기복      {hilly:.0f} kWh  (+{(hilly / flat - 1) * 100:.0f}%)")
print("  → 회생 회수율이 60%라 오르내림이 상쇄되지 않습니다. 평균 구배로는 이 손실이 안 잡힙니다.")

평지 정속 전비 (kWh/100km)   ※ 공인치 98

속도 km/h    80     90     100
총중량                         
25t      105.1  118.8  134.3
30t      114.7  128.4  143.9
40t      133.9  147.7  163.2
65t      182.0  195.8  211.2

→ 공인 98 kWh/100km 는 25~30t·80km/h 조건입니다.
   40t 만재 90km/h 면 148 kWh/100km 로 봐야 합니다.

100 km 구간, 순고도차 0:
  완전 평지        148 kWh
  ±500 m 기복      179 kWh  (+21%)
  → 회생 회수율이 60%라 오르내림이 상쇄되지 않습니다. 평균 구배로는 이 손실이 안 잡힙니다.


## 2. ML 보정계수

2번 노트북에서 학습한 모델을 **비율로만** 씁니다. 기준 조건 대비 몇 배인지가 보정계수입니다.
절대값을 안 쓰기 때문에 4.1배 스케일 차이가 상쇄됩니다.

In [3]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import Ridge

EV_CSV = "data/ev_energy_consumption.csv"
TARGET = "energy_consumption_kwhper100km"

ev = pd.read_csv(EV_CSV, encoding="utf-8-sig")
FEATURES = [c for c in ev.columns if c != TARGET]

ml = make_pipeline(StandardScaler(), PolynomialFeatures(2, include_bias=False), Ridge(alpha=1.0))
ml.fit(ev[FEATURES], ev[TARGET])
print(f"ML 학습 완료 (train R2 = {ml.score(ev[FEATURES], ev[TARGET]):.4f})")

# 보정계수의 기준점 — 데이터셋 중앙값 조건
REF = {c: float(ev[c].median()) for c in FEATURES}
REF_PRED = float(ml.predict(pd.DataFrame([REF]))[0])
print(f"기준 조건 예측치: {REF_PRED:.2f} kWh/100km")


def hvac_from_temp(temp_c):
    """기온에서 공조 부하 추정. 쾌적구간 18~24도에서 최소, 양쪽으로 증가."""
    return float(np.clip(abs(temp_c - 21) * 0.18, 0, 5))


def ml_correction(speed_kmh, dist_km, temp_c=AMBIENT_C, style=DRIVING_STYLE):
    """기준 조건 대비 배율. 물리 모델이 이미 다루는 구배·적재는 기준값 고정."""
    row = dict(REF)
    row.update({
        "speed_kmh": float(np.clip(speed_kmh, ev["speed_kmh"].min(), ev["speed_kmh"].max())),
        "trip_distance_km": float(np.clip(dist_km, ev["trip_distance_km"].min(),
                                          ev["trip_distance_km"].max())),
        "ambient_temp_C": float(np.clip(temp_c, ev["ambient_temp_C"].min(),
                                        ev["ambient_temp_C"].max())),
        "hvac_power_kw": hvac_from_temp(temp_c),
        "driving_style_index": float(style),
    })
    return float(ml.predict(pd.DataFrame([row]))[0]) / REF_PRED


for t_ in [-10, 0, 10, 21, 30, 40]:
    print(f"  기온 {t_:+3d}°C → 보정계수 {ml_correction(90, 80, t_):.3f}")

ML 학습 완료 (train R2 = 0.9529)
기준 조건 예측치: 23.44 kWh/100km
  기온 -10°C → 보정계수 1.232
  기온  +0°C → 보정계수 1.112
  기온 +10°C → 보정계수 0.998
  기온 +21°C → 보정계수 0.903
  기온 +30°C → 보정계수 0.973
  기온 +40°C → 보정계수 1.076


## 3. 구간별 전비 예측

1번 노트북의 `df_segments` 를 읽어 구간마다 전비를 계산합니다.

**`상승_m` / `하강_m` 컬럼이 없으면 평지로 처리됩니다.** 구배는 전비 기여도 1위 인자라 반드시 붙이세요 —
고도를 안 넣으면 산악 구간에서 크게 빗나갑니다.

In [4]:
SEG_CSV = "busan_incheon_route_segments.csv"
seg = pd.read_csv(SEG_CSV, encoding="utf-8-sig")

for col in ["상승_m", "하강_m"]:
    if col not in seg.columns:
        seg[col] = 0.0
if (seg["상승_m"] == 0).all():
    print("경고: 고도 컬럼(상승_m / 하강_m)이 없어 평지로 처리합니다.")
    print("      전비 기여도 1위 인자가 빠진 상태이니 산악 구간은 과소평가됩니다.\n")


def predict_segment(row, mass_kg=GCW_KG, temp_c=AMBIENT_C, style=DRIVING_STYLE):
    base = segment_energy_kwh(row["구간거리_km"], row["평균속도_kmh"],
                              row["상승_m"], row["하강_m"], mass_kg)
    return base * ml_correction(row["평균속도_kmh"], row["구간거리_km"], temp_c, style)


seg["소모_kWh"] = [round(predict_segment(r), 1) for _, r in seg.iterrows()]
seg["전비_kWh100km"] = (seg["소모_kWh"] / seg["구간거리_km"] * 100).round(1)

print(f"총 소모 예측: {seg['소모_kWh'].sum():.0f} kWh  "
      f"(가용 {BATT_USABLE_KWH:.0f} kWh 의 {seg['소모_kWh'].sum() / BATT_USABLE_KWH * 100:.0f}%)")
print(f"평균 전비: {seg['소모_kWh'].sum() / seg['구간거리_km'].sum() * 100:.1f} kWh/100km")
seg[["출발", "도착", "구간거리_km", "평균속도_kmh", "전비_kWh100km", "소모_kWh"]]

경고: 고도 컬럼(상승_m / 하강_m)이 없어 평지로 처리합니다.
      전비 기여도 1위 인자가 빠진 상태이니 산악 구간은 과소평가됩니다.



ValueError: Input X contains NaN.
PolynomialFeatures does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

## 4. SOC 시뮬레이션 및 충전 계획

각 구간을 주행하며 SOC를 깎고, 다음 구간을 못 버티면 **직전 휴게소에서 충전**합니다.
충전 가능 여부는 1번 노트북의 `df_rest_ev.충전기수` 로 판단합니다.

In [ ]:
REST_CSV = "busan_incheon_rest_areas_ev.csv"
try:
    rest = pd.read_csv(REST_CSV, encoding="utf-8-sig")
    can_charge = dict(zip(rest["name"], rest["충전기수"] > 0))
    print(f"휴게소 {len(rest)}개 중 충전 가능 {sum(can_charge.values())}개")
except FileNotFoundError:
    can_charge = {}
    print("휴게소 파일이 없어 모든 지점에서 충전 가능하다고 가정합니다.")


def simulate(seg, soc_start=SOC_START, soc_min=SOC_MIN, soc_to=SOC_CHARGE_TO,
             charger_kw=CHG_MAX_KW, charge_eff=0.73, can_charge=None):
    """SOC를 추적하며 충전 계획을 세운다.

    핵심은 앞을 내다보는 것입니다. 충전기가 있는 휴게소에 도착할 때마다
    '다음 충전 가능 휴게소(없으면 목적지)까지 갈 수 있는가'를 보고 충전 여부를 정합니다.
    한 구간만 보면 충전기 없는 휴게소를 지나친 뒤에야 부족을 깨닫게 됩니다.
    """
    can_charge = {} if can_charge is None else can_charge
    n = len(seg)
    e = seg["소모_kWh"].values
    avail = BATT_USABLE_KWH

    # 각 구간 도착지에서 충전 가능한가 (목적지는 제외)
    has_charger = [bool(can_charge.get(seg["도착"].iat[i], True)) for i in range(n)]
    if n:
        has_charger[n - 1] = False

    soc, log, charges, total_min = soc_start, [], 0, 0.0
    for i in range(n):
        soc_after = soc - e[i] / avail * 100
        row = {"구간": f"{seg['출발'].iat[i]} → {seg['도착'].iat[i]}",
               "거리_km": seg["구간거리_km"].iat[i], "소모_kWh": e[i],
               "출발SOC": round(soc, 1), "도착SOC": round(soc_after, 1), "조치": ""}

        if soc_after < soc_min:
            row["조치"] = f"!! 도달 불가 (SOC {soc_after:.0f}%)"
            log.append(row)
            soc = soc_after
            continue
        soc = soc_after

        if has_charger[i]:
            # 다음 충전 가능 지점까지(없으면 목적지까지)의 소요를 미리 계산
            nxt = next((m for m in range(i + 1, n - 1) if has_charger[m]), None)
            span = e[i + 1: (nxt + 1 if nxt is not None else n)].sum()
            if soc - span / avail * 100 < soc_min:
                kwh = (soc_to - soc) / 100 * avail
                mins = kwh / (charger_kw * charge_eff) * 60
                row["조치"] = f"충전 {soc:.0f}→{soc_to:.0f}% ({kwh:.0f} kWh, {mins:.0f}분)"
                soc, charges, total_min = soc_to, charges + 1, total_min + mins
        log.append(row)

    return pd.DataFrame(log), soc, charges, total_min


sim, soc_end, n_chg, chg_min = simulate(seg, can_charge=can_charge)
ok = soc_end >= SOC_MIN and not sim["조치"].str.contains("도달 불가").any()
print(f"\n{'완주 가능' if ok else '완주 불가'} | 도착 SOC {soc_end:.1f}% | "
      f"충전 {n_chg}회 ({chg_min:.0f}분)")
sim

In [ ]:
# 시나리오 비교 — 조건이 바뀌면 충전 횟수가 어떻게 달라지는지
scen = []
for label, mass, temp in [
    ("공차 25t / 봄가을 15°C", 25_000, 15),
    ("30t / 겨울 0°C", 30_000, 0),
    ("40t 만재 / 겨울 -5°C", 40_000, -5),
    ("40t 만재 / 여름 33°C", 40_000, 33),
]:
    s = seg.copy()
    s["소모_kWh"] = [predict_segment(r, mass, temp) for _, r in s.iterrows()]
    _, soc_e, n, mins = simulate(s, can_charge=can_charge)
    scen.append({"시나리오": label,
                 "평균전비_kWh100km": round(s["소모_kWh"].sum() / s["구간거리_km"].sum() * 100, 1),
                 "총소모_kWh": round(s["소모_kWh"].sum()),
                 "충전횟수": n, "충전시간_분": round(mins),
                 "도착SOC_%": round(soc_e, 1),
                 "완주": "O" if soc_e >= SOC_MIN else "X"})

pd.DataFrame(scen)

## 5. 현실 점검 — 충전 인프라

시뮬레이션이 "1회 충전이면 완주"라고 해도, **그 충전기가 실제로 있느냐**는 별개입니다.

**충전 출력.** 20→80%(276 kWh)를 65분에 넣으려면 평균 255 kW가 필요합니다.
2022년 기준 전국 고속도로 휴게소 충전기 860기 중 **82%가 100 kW 이하**였고,
200 kW 이상은 18%뿐이었습니다. 100 kW 충전기라면 같은 양을 넣는 데 **약 3시간 15분**이 걸립니다.

**물리적 접근성.** 이게 더 큰 문제입니다. 트랙터+트레일러는 전장 16.5 m 이상인데,
휴게소 충전 구획은 승용차 기준으로 설계돼 있습니다. 충전기가 있어도 **진입과 주차가 안 되면 무의미**합니다.
1번 노트북의 `충전기수` 는 "반경 안에 충전기가 존재한다"는 뜻이지 "대형 트럭이 쓸 수 있다"가 아닙니다.

→ 실제 운행 계획을 세우실 거면 후보 휴게소에 대해 **화물차 전용 충전 구획 유무를 개별 확인**해야 합니다.
   이건 API로 안 나오고 현장 확인이나 사업자 문의가 필요합니다.

In [ ]:
print("충전기 출력별 20→80% (276 kWh) 소요시간\n")
for kw, eff, note in [(350, 0.73, "제원상 65분"),
                      (200, 0.85, "휴게소 상위 18%"),
                      (100, 0.85, "휴게소 82%가 이 이하"),
                      (50, 0.85, "구형")]:
    print(f"  {kw:3d} kW → {276 / (kw * eff) * 60:5.0f}분   ({note})")

print("\n1회 충전 시나리오에서 충전 지점이 놓일 수 있는 누적거리 구간:\n")
ROUTE_KM = seg["구간거리_km"].sum()
for label, k in [("공인치 25~30t", oem_kwh100),
                 ("30t 90km/h", kwh_per_100km(90, 30_000)),
                 ("40t 만재 90km/h", kwh_per_100km(90, 40_000))]:
    leg1 = BATT_USABLE_KWH * (100 - SOC_MIN) / 100 / k * 100
    cyc = BATT_USABLE_KWH * (SOC_CHARGE_TO - SOC_MIN) / 100 / k * 100
    lo, hi = max(0.0, ROUTE_KM - cyc), min(ROUTE_KM, leg1)
    if hi < lo:
        print(f"  {label:16s} {k:5.1f} → 1회로 불가, 2회 필요")
    else:
        print(f"  {label:16s} {k:5.1f} → 누적 {lo:5.1f} ~ {hi:5.1f} km  (폭 {hi - lo:5.1f} km)")
print("\n폭이 좁을수록 그 구간에 충전 가능한 휴게소가 실제로 있는지가 결정적입니다.")

---

## 정리 — 이 모델의 신뢰 구간

**믿을 만한 것**
- 물리 모델의 절대 수준. 종방향 동역학은 검증된 식이고, 공인 전비를 25~30t 조건으로 잘 재현합니다.
- 시나리오 간 **상대 비교**. 만재 대비 공차, 겨울 대비 봄가을이 몇 배인지.

**믿기 어려운 것**
- ML 보정계수의 절대 정확도. 승용차 합성 데이터에서 뽑은 상대 민감도를 트럭에 전이한 것입니다.
- 구배 0% 가정 하의 구간별 전비. 고도 데이터를 붙이기 전까지는 산악 구간이 과소평가됩니다.
- 회생제동 회수율 0.60. 차량·노면·운전에 따라 0.4~0.7로 흔들립니다.

**다음에 할 것 (효과 큰 순서)**
1. **고도(DEM) 연동** — 구배는 전비 기여도 1위인데 지금 비어 있습니다.
2. **기상청 ASOS 연동** — 구간별 실제 기온. 지금은 단일 상수입니다.
3. **후보 휴게소 화물차 충전 가능 여부 현장 확인** — 시뮬레이션 결과를 무의미하게 만들 수 있는 유일한 변수입니다.
4. 실측 트럭 주행 데이터 확보 시 보정계수 재학습.